# BÀI TẬP LỚN HỌC MÁY - BÁO CÁO MILESTONE 1
## 01. Phân Tích Dữ Liệu Khám Phá (EDA) & Thiết Kế Pipeline Tiền Xử Lý Chống Data Leakage
---
- **Bộ dữ liệu:** Framingham Heart Study (`data/framingham.csv`)
- **Mục tiêu:** Sàng lọc & Dự báo nguy cơ mắc bệnh tim mạch vành trong 10 năm (`TenYearCHD`)
- **Thành viên phụ trách:** Thành viên 1 (Nhóm trưởng) & Thành viên 3
- **Mốc chấm:** Milestone 1 (Checkpoint Buổi 4) - Bám sát Nhiệm vụ STT 1, 3, 4, 5 trong Bảng Phân Công

In [ ]:
# 1. Import các thư viện phân tích và hệ thống
import os
import sys
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Thêm thư mục gốc vào sys.path để import trực tiếp từ src
sys.path.append(os.path.abspath('..'))
from src.features import (
    create_features,
    get_preprocessor_pipeline,
    load_and_split_data,
    NUMERIC_FEATURES,
    CATEGORICAL_FEATURES,
    TARGET_COLUMN
)

# Cấu hình giao diện biểu đồ
plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', None)
print('Imports và cấu hình thành công!')

## 1. Tải Dữ Liệu & Kiểm Tra Cấu Trúc Tổng Quát
Sử dụng đường dẫn tương đối chuẩn `../data/framingham.csv` theo quy ước của nhóm.

In [ ]:
df = pd.read_csv('../data/framingham.csv')
print(f'Kích thước bộ dữ liệu ban đầu: {df.shape[0]} dòng x {df.shape[1]} cột')
df.head()

In [ ]:
# Thống kê mô tả các biến số
df.describe().T

## 2. Kiểm Tra Dữ Liệu Thiếu (Missing Values)
Phát hiện các cột có dữ liệu thiếu để thiết kế chiến lược điền khuyết (Imputation) phù hợp.

In [ ]:
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Số lượng khuyết': missing, 'Tỷ lệ (%)': missing_pct})
missing_df = missing_df[missing_df['Số lượng khuyết'] > 0].sort_values('Số lượng khuyết', ascending=False)

print(f'Tổng số giá trị bị khuyết: {df.isnull().sum().sum()}')
print(f'Số lượng cột bị khuyết: {len(missing_df)} / {df.shape[1]}')
missing_df

## 3. Trực Quan Hóa Dữ Liệu Khám Phá (EDA)
Phân tích phân phối lớp mục tiêu `TenYearCHD`, các biến số lâm sàng và mối tương quan giữa các đặc trưng.

In [ ]:
# 3.1. Phân phối biến mục tiêu TenYearCHD (Mất cân bằng dữ liệu)
fig, ax = plt.subplots(figsize=(6, 4), dpi=300)
counts = df[TARGET_COLUMN].value_counts()
colors = ['#2b5c8f', '#d9534f']
bars = ax.bar(['0: Không mắc CHD', '1: Có mắc CHD'], counts.values, color=colors, width=0.5)

for bar in bars:
    yval = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, yval + 50, f'{yval:,} ({yval/len(df):.1%})', 
            ha='center', va='bottom', fontweight='bold')

ax.set_title('Phân phối biến mục tiêu TenYearCHD (Mất cân bằng lớp)', fontsize=12, pad=15)
ax.set_ylabel('Số lượng mẫu')
ax.set_ylim(0, 4200)
plt.tight_layout()
plt.show()

In [ ]:
# 3.2. Phân phối tuổi, huyết áp, cholesterol theo nhóm mắc bệnh
fig, axes = plt.subplots(1, 3, figsize=(16, 5), dpi=300)
features_plot = [('age', 'Tuổi (Age)'), ('sysBP', 'Huyết áp tâm thu (sysBP)'), ('totChol', 'Tổng Cholesterol (totChol)')]

for i, (col, label) in enumerate(features_plot):
    sns.kdeplot(data=df, x=col, hue=TARGET_COLUMN, common_norm=False, 
                palette=['#2b5c8f', '#d9534f'], fill=True, alpha=0.4, ax=axes[i])
    axes[i].set_title(f'Phân phối {label} theo TenYearCHD', fontsize=11)
    axes[i].set_xlabel(label)
    axes[i].legend(['Có mắc CHD (1)', 'Không mắc CHD (0)'])

plt.tight_layout()
plt.show()

In [ ]:
# 3.3. Ma trận tương quan (Correlation Matrix Heatmap)
fig, ax = plt.subplots(figsize=(12, 10), dpi=300)
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1, 
            center=0, square=True, linewidths=.5, cbar_kws={'shrink': .8}, ax=ax)
ax.set_title('Ma trận hệ số tương quan Pearson giữa các đặc trưng', fontsize=13, pad=15)
plt.tight_layout()
plt.show()

> **Nhận xét quan trọng:**
> 1. **Tương quan mạnh giữa huyết áp tâm thu (`sysBP`) và tâm trương (`diaBP`)**: Hệ số tương quan $r = 0.78$. Cả hai phản ánh tình trạng áp lực lòng mạch. Do đó, việc kết hợp thành các chỉ số y khoa như **Áp lực mạch (Pulse Pressure)** và **Huyết áp trung bình (MAP)** là có cơ sở khoa học vững chắc.
> 2. **Tương quan giữa `currentSmoker` và `cigsPerDay`** ($r = 0.77$): Người hút thuốc có nguy cơ cao hơn khi tuổi tác tăng lên. Đặc trưng tích hợp `age_cigs_interaction` sẽ nắm bắt hiệu ứng tương tác này.

---
## 4. Kỹ Thuật Tạo Đặc Trưng Mới (Feature Engineering - STT 5)
Dựa trên kiến thức chuyên môn tim mạch và phân tích EDA ở trên, ta tạo 3 đặc trưng mới:
1. **`pulse_pressure` (Áp lực mạch):**
   $$\text{pulse\_pressure} = \text{sysBP} - \text{diaBP}$$
   *Ý nghĩa y học:* Độ chênh lệch phản ánh độ cứng thành động mạch. Áp lực mạch cao (> 60 mmHg) là yếu tố độc lập dự báo xơ vữa động mạch và đột quỵ.
2. **`MAP` (Mean Arterial Pressure - Huyết áp động mạch trung bình):**
   $$\text{MAP} = \text{diaBP} + \frac{\text{sysBP} - \text{diaBP}}{3}$$
   *Ý nghĩa y học:* Áp lực tưới máu thực tế liên tục đến các cơ quan quan trọng, toàn diện hơn huyết áp đơn lẻ.
3. **`age_cigs_interaction` (Tương tác Tuổi x Lượng thuốc lá):**
   $$\text{age\_cigs\_interaction} = \text{age} \times \text{cigsPerDay}$$
   *Ý nghĩa y học:* Mức độ phơi nhiễm độc chất khói thuốc theo thời gian tích lũy theo số năm tuổi.

In [ ]:
# Thử nghiệm hàm create_features từ module src/features.py
df_featured = create_features(df)
print('3 đặc trưng mới đã được bổ sung vào DataFrame:')
df_featured[['sysBP', 'diaBP', 'pulse_pressure', 'MAP', 'age', 'cigsPerDay', 'age_cigs_interaction']].head()

---
## 5. Chia Tập Dữ Liệu Train / Test (Tuân Thủ Tuyệt Đối Quy Ước 1 & 2)
> ⚠️ **LƯU Ý BẮT BUỘC ĐỂ KHÔNG BỊ TRỪ ĐIỂM (Data Leakage):**
> - BẮT BUỘC thực hiện chia tập Train (80%) và Test (20%) bằng `train_test_split` trước khi thực hiện bất kỳ phép `fit` nào của Imputer hoặc Scaler.
> - Dùng `stratify=y` và `random_state=42` để cố định dữ liệu và bảo toàn tỷ lệ nhãn cho cả 3 máy.

In [ ]:
# Gọi hàm load_and_split_data chuẩn từ module src/features.py
X_train, X_test, y_train, y_test = load_and_split_data(
    data_path='../data/framingham.csv',
    test_size=0.2,
    random_state=42
)

print(f'Kích thước X_train: {X_train.shape} | y_train: {y_train.shape}')
print(f'Kích thước X_test:  {X_test.shape}  | y_test:  {y_test.shape}')
print(f'Tỷ lệ nhãn dương (TenYearCHD=1) trên Train: {y_train.mean():.4f}')
print(f'Tỷ lệ nhãn dương (TenYearCHD=1) trên Test:  {y_test.mean():.4f}')

---
## 6. Đóng Gói Bộ Tiền Xử Lý Pipeline (ColumnTransformer) Chống Rò Rỉ Dữ Liệu
Toàn bộ quy trình tiền xử lý được đóng gói trong một `Pipeline` duy nhất gồm:
1. **`HeartDiseaseFeatureEngineer`:** Tự động tính toán 3 đặc trưng mới trên dữ liệu.
2. **`ColumnTransformer`:**
   - **11 Đặc trưng số:** `SimpleImputer(strategy='median')` -> `StandardScaler()`
   - **7 Đặc trưng phân loại:** `SimpleImputer(strategy='most_frequent')`

**Cơ chế chống Data Leakage:**
- `pipeline.fit(X_train)`: Chỉ học Median và Mean/Std từ `X_train`.
- `pipeline.transform(X_test)`: Dùng nguyên các thông số học được từ `X_train` áp dụng lên `X_test`.

In [ ]:
# Khởi tạo Pipeline tiền xử lý hoàn chỉnh từ src/features.py
preprocessor_pipeline = get_preprocessor_pipeline()
print('Pipeline đã khởi tạo thành công:')
preprocessor_pipeline

In [ ]:
# FIT CHỈ TRÊN TẬP TRAIN (Chống rò rỉ dữ liệu)
preprocessor_pipeline.fit(X_train, y_train)

# Transform trên cả Train và Test độc lập
X_train_processed = preprocessor_pipeline.transform(X_train)
X_test_processed = preprocessor_pipeline.transform(X_test)

print(f'X_train sau tiền xử lý: {X_train_processed.shape}')
print(f'X_test sau tiền xử lý:  {X_test_processed.shape}')
print(f'Số giá trị NaN còn lại trong X_train: {np.isnan(X_train_processed).sum()}')
print(f'Số giá trị NaN còn lại trong X_test:  {np.isnan(X_test_processed).sum()}')

## 7. Xuất & Lưu Pipeline Hoàn Chỉnh Cho Thành Viên 2 (Phụ Trách Mô Hình)
Lưu `preprocessor_pipeline` đã fit vào thư mục `models/` để Thành viên 2 (Mô hình Baseline & Multi-Model) lắp trực tiếp vào mô hình huấn luyện.

In [ ]:
import joblib

os.makedirs('../models', exist_ok=True)
pipeline_save_path = '../models/preprocessor_pipeline.joblib'
joblib.dump(preprocessor_pipeline, pipeline_save_path)
print(f'Đã lưu thành công bộ tiền xử lý Pipeline tại: {pipeline_save_path}')
print('Hoàn tất toàn bộ yêu cầu của STT 5 (Mốc 1 Checkpoint 1)!')